In [1]:
import wandb
print(wandb.__file__)     # should point to your site-packages, NOT your project folder
print(dir(wandb))         # should include 'init'


None
['__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [1]:
import os
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from models import ImageClassifier, FCNet
from datasets import get_metadata, get_imagenet_stats
import json

In [3]:
base_path = r'results\multi_label_experiment_2025_09_14_21-03-27_pascal_no_constraints_fine_tuned_from_linear/'
logic_path = r'results\multi_label_experiment_2025_09_20_16-21-18_pascal_no_constraints_fine_tuned_from_linear_improved_loss_update/'

In [5]:
# Load params from a previous run
with open(base_path + 'params.json', 'r') as f:
    BASE_P = json.load(f)
    
# Load params from a previous run
with open(logic_path + 'params.json', 'r') as f:
    LOGIC_P = json.load(f)

In [6]:
# # Match saved model exactly
b_linear_classifier = FCNet(BASE_P['feat_dim'], BASE_P['num_classes'])
b_model = ImageClassifier(BASE_P, model_linear_classifier=b_linear_classifier)
b_model.load_state_dict(torch.load(base_path + 'best_model_state_f.pt', map_location='cpu'))
b_model.eval()

l_linear_classifier = FCNet(LOGIC_P['feat_dim'], LOGIC_P['num_classes'])
l_model = ImageClassifier(LOGIC_P, model_linear_classifier=l_linear_classifier) 
l_model.load_state_dict(torch.load(logic_path + 'best_model_state_f.pt', map_location='cpu'))
l_model.eval()

initializing image classifier
feature extractor: imagenet pretrained


c:\Users\ibrah\Workspace\T\spmll\venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\ibrah\Workspace\T\spmll\venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


feature extractor trainable
linear classifier layer: specified by user
initializing image classifier
feature extractor: imagenet pretrained


C:\Users\ibrah\AppData\Local\Temp\ipykernel_9696\3199334528.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  b_model.load_state_dict(torch.load(base_path + 'best_model_st

feature extractor trainable
linear classifier layer: specified by user


C:\Users\ibrah\AppData\Local\Temp\ipykernel_9696\3199334528.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  l_model.load_state_dict(torch.load(logic_path + 'best_model_s

ImageClassifier(
  (feature_extractor): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
        

In [7]:
val_images = np.load(r"data/pascal/formatted_val_images.npy")
val_labels = np.load(r"data/pascal/formatted_val_labels.npy")
val_labels_obs = np.load(r"data/pascal/formatted_val_labels_obs.npy")
val_images[0], val_labels[0], val_labels_obs[0]

(np.str_('2008_000002.jpg'),
 array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 1., 0., 0., 1.], dtype=float32),
 array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 1., 0., 0., 0.], dtype=float32))

In [8]:
mean, std = get_imagenet_stats()
transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Load category labels
category_map = {
    0: 'aeroplane', 1: 'bicycle', 2: 'bird', 3: 'boat', 4: 'bottle',
    5: 'bus', 6: 'car', 7: 'cat', 8: 'chair', 9: 'cow',
    10: 'diningtable', 11: 'dog', 12: 'horse', 13: 'motorbike', 14: 'person',
    15: 'pottedplant', 16: 'sheep', 17: 'sofa', 18: 'train', 19: 'tvmonitor', 20: "vehicle", 21: "animal", 22: "indoor"
}

In [9]:
class_names = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor', "vehicle", "animal", "indoor"
]

In [10]:
import os
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_fscore_support, f1_score, precision_score, recall_score,
    accuracy_score, hamming_loss
)

# ---- Inputs you already have ----
# class_names = [...]
# val_images, val_labels, val_labels_obs = np.load(...)
# transform, l_model defined elsewhere
print("Using base model for evaluation...")
IMG_DIR = r"data/pascal/VOCdevkit/VOC2012/JPEGImages/"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
b_model = b_model.to(DEVICE).eval()

# ---------- 1) Get probabilities for ALL validation images ----------
def predict_all_probs(img_names, batch_size=16, threshold=None):
    """
    Returns:
        y_scores: (N, C) float array of probabilities
    """
    N = len(img_names)
    # Peek one image to get C
    first_img = Image.open(os.path.join(IMG_DIR, img_names[0])).convert("RGB")
    with torch.no_grad():
        C = b_model(transform(first_img).unsqueeze(0).to(DEVICE)).shape[-1]

    y_scores = np.zeros((N, C), dtype=np.float32)

    # Simple batching without DataLoader
    for start in tqdm(range(0, N, batch_size), desc="Predicting"):
        end = min(start + batch_size, N)
        batch_imgs = []
        for i in range(start, end):
            p = os.path.join(IMG_DIR, img_names[i])
            img = Image.open(p).convert("RGB")
            batch_imgs.append(transform(img))
        batch_tensor = torch.stack(batch_imgs, dim=0).to(DEVICE)

        with torch.no_grad():
            logits = b_model(batch_tensor)
            probs = torch.sigmoid(logits).cpu().numpy()
        y_scores[start:end] = probs

    return y_scores

# Ensure filenames are strings (sometimes np.load yields bytes)
val_files = [fn.decode() if isinstance(fn, bytes) else str(fn) for fn in val_images]
y_true = val_labels.astype(int)             # shape (N, C)
y_scores = predict_all_probs(val_files, batch_size=16)

# ---------- 2) Compute metrics ----------
def multilabel_metrics(y_true, y_scores, class_names, threshold=0.5):
    """
    Computes:
      - Per-class: AP, AUROC, Precision, Recall, F1 (at threshold)
      - Overall: mAP (macro), AP-micro, AUROC-macro, AUROC-micro,
                 Precision/Recall/F1 (micro & macro at threshold),
                 Subset Accuracy, Hamming Loss
    """
    N, C = y_true.shape
    assert C == len(class_names), "class_names length must match labels"

    # Per-class AP & AUROC (use probabilities)
    ap_per_class = []
    auroc_per_class = []
    for c in range(C):
        y_c = y_true[:, c]
        s_c = y_scores[:, c]
        # average_precision_score works even with class imbalance
        ap_c = average_precision_score(y_c, s_c) if (y_c.max() != y_c.min()) else np.nan
        # roc_auc_score requires both classes present
        try:
            auc_c = roc_auc_score(y_c, s_c)
        except ValueError:
            auc_c = np.nan
        ap_per_class.append(ap_c)
        auroc_per_class.append(auc_c)

    # Threshold to get binary predictions
    y_pred = (y_scores >= threshold).astype(int)

    # Per-class P/R/F1 at threshold
    prec_c, rec_c, f1_c, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    # Overall (micro/macro)
    map_macro = np.nanmean(ap_per_class)
    ap_micro  = average_precision_score(y_true, y_scores, average="micro")
    auroc_macro = np.nanmean(auroc_per_class)
    # For AUROC micro, flatten
    try:
        auroc_micro = roc_auc_score(y_true.ravel(), y_scores.ravel())
    except ValueError:
        auroc_micro = np.nan

    p_micro = precision_score(y_true, y_pred, average="micro", zero_division=0)
    r_micro = recall_score(y_true, y_pred, average="micro", zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)

    p_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    r_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

    subset_acc = accuracy_score(y_true, y_pred)  # exact match ratio
    hamming = hamming_loss(y_true, y_pred)

    # Build per-class table
    per_class_df = pd.DataFrame({
        "Class": class_names,
        "AP": ap_per_class,
        "AUROC": auroc_per_class,
        "Precision@{:.2f}".format(threshold): prec_c,
        "Recall@{:.2f}".format(threshold): rec_c,
        "F1@{:.2f}".format(threshold): f1_c,
        "Support": y_true.sum(axis=0)
    })

    # Overall summary
    overall = {
        "mAP (macro mean AP)": map_macro,
        "AP (micro)": ap_micro,
        "AUROC (macro mean)": auroc_macro,
        "AUROC (micro)": auroc_micro,
        "Precision (micro)@{:.2f}".format(threshold): p_micro,
        "Recall (micro)@{:.2f}".format(threshold): r_micro,
        "F1 (micro)@{:.2f}".format(threshold): f1_micro,
        "Precision (macro)@{:.2f}".format(threshold): p_macro,
        "Recall (macro)@{:.2f}".format(threshold): r_macro,
        "F1 (macro)@{:.2f}".format(threshold): f1_macro,
        "Subset Accuracy": subset_acc,
        "Hamming Loss": hamming,
    }

    overall_df = pd.DataFrame(list(overall.items()), columns=["Metric", "Value"])
    return per_class_df, overall_df, y_pred

per_class_df, overall_df, y_pred = multilabel_metrics(y_true, y_scores, class_names, threshold=0.5)

print("\n=== Overall Metrics ===")
print(overall_df.to_string(index=False))

print("\n=== Per-class Metrics ===")
print(per_class_df.sort_values("AP", ascending=False).to_string(index=False))


Using base model for evaluation...


Predicting: 100%|██████████| 364/364 [01:09<00:00,  5.23it/s]



=== Overall Metrics ===
                Metric    Value
   mAP (macro mean AP) 0.872811
            AP (micro) 0.919835
    AUROC (macro mean) 0.972352
         AUROC (micro) 0.978146
Precision (micro)@0.50 0.950181
   Recall (micro)@0.50 0.720718
       F1 (micro)@0.50 0.819694
Precision (macro)@0.50 0.939310
   Recall (macro)@0.50 0.625525
       F1 (macro)@0.50 0.731861
       Subset Accuracy 0.538382
          Hamming Loss 0.032704

=== Per-class Metrics ===
      Class       AP    AUROC  Precision@0.50  Recall@0.50  F1@0.50  Support
     animal 0.988943 0.990798        0.990865     0.880611 0.932491     2094
  aeroplane 0.984543 0.998677        0.981132     0.909621 0.944024      343
    vehicle 0.971354 0.982642        0.936249     0.898837 0.917161     1977
      train 0.966905 0.996731        1.000000     0.778598 0.875519      271
        cat 0.964921 0.990766        0.967880     0.835490 0.896825      541
       bird 0.957376 0.984228        0.996564     0.783784 0.877458   

In [11]:
import os
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_fscore_support, f1_score, precision_score, recall_score,
    accuracy_score, hamming_loss
)

# ---- Inputs you already have ----
# class_names = [...]
# val_images, val_labels, val_labels_obs = np.load(...)
# transform, l_model defined elsewhere
print("Using logic model for evaluation...")
IMG_DIR = r"data/pascal/VOCdevkit/VOC2012/JPEGImages/"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
l_model = l_model.to(DEVICE).eval()

# ---------- 1) Get probabilities for ALL validation images ----------
def predict_all_probs(img_names, batch_size=16, threshold=None):
    """
    Returns:
        y_scores: (N, C) float array of probabilities
    """
    N = len(img_names)
    # Peek one image to get C
    first_img = Image.open(os.path.join(IMG_DIR, img_names[0])).convert("RGB")
    with torch.no_grad():
        C = l_model(transform(first_img).unsqueeze(0).to(DEVICE)).shape[-1]

    y_scores = np.zeros((N, C), dtype=np.float32)

    # Simple batching without DataLoader
    for start in tqdm(range(0, N, batch_size), desc="Predicting"):
        end = min(start + batch_size, N)
        batch_imgs = []
        for i in range(start, end):
            p = os.path.join(IMG_DIR, img_names[i])
            img = Image.open(p).convert("RGB")
            batch_imgs.append(transform(img))
        batch_tensor = torch.stack(batch_imgs, dim=0).to(DEVICE)

        with torch.no_grad():
            logits = l_model(batch_tensor)
            probs = torch.sigmoid(logits).cpu().numpy()
        y_scores[start:end] = probs

    return y_scores

# Ensure filenames are strings (sometimes np.load yields bytes)
val_files = [fn.decode() if isinstance(fn, bytes) else str(fn) for fn in val_images]
y_true = val_labels.astype(int)             # shape (N, C)
y_scores = predict_all_probs(val_files, batch_size=16)

# ---------- 2) Compute metrics ----------
def multilabel_metrics(y_true, y_scores, class_names, threshold=0.5):
    """
    Computes:
      - Per-class: AP, AUROC, Precision, Recall, F1 (at threshold)
      - Overall: mAP (macro), AP-micro, AUROC-macro, AUROC-micro,
                 Precision/Recall/F1 (micro & macro at threshold),
                 Subset Accuracy, Hamming Loss
    """
    N, C = y_true.shape
    assert C == len(class_names), "class_names length must match labels"

    # Per-class AP & AUROC (use probabilities)
    ap_per_class = []
    auroc_per_class = []
    for c in range(C):
        y_c = y_true[:, c]
        s_c = y_scores[:, c]
        # average_precision_score works even with class imbalance
        ap_c = average_precision_score(y_c, s_c) if (y_c.max() != y_c.min()) else np.nan
        # roc_auc_score requires both classes present
        try:
            auc_c = roc_auc_score(y_c, s_c)
        except ValueError:
            auc_c = np.nan
        ap_per_class.append(ap_c)
        auroc_per_class.append(auc_c)

    # Threshold to get binary predictions
    y_pred = (y_scores >= threshold).astype(int)

    # Per-class P/R/F1 at threshold
    prec_c, rec_c, f1_c, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    # Overall (micro/macro)
    map_macro = np.nanmean(ap_per_class)
    ap_micro  = average_precision_score(y_true, y_scores, average="micro")
    auroc_macro = np.nanmean(auroc_per_class)
    # For AUROC micro, flatten
    try:
        auroc_micro = roc_auc_score(y_true.ravel(), y_scores.ravel())
    except ValueError:
        auroc_micro = np.nan

    p_micro = precision_score(y_true, y_pred, average="micro", zero_division=0)
    r_micro = recall_score(y_true, y_pred, average="micro", zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)

    p_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    r_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

    subset_acc = accuracy_score(y_true, y_pred)  # exact match ratio
    hamming = hamming_loss(y_true, y_pred)

    # Build per-class table
    per_class_df = pd.DataFrame({
        "Class": class_names,
        "AP": ap_per_class,
        "AUROC": auroc_per_class,
        "Precision@{:.2f}".format(threshold): prec_c,
        "Recall@{:.2f}".format(threshold): rec_c,
        "F1@{:.2f}".format(threshold): f1_c,
        "Support": y_true.sum(axis=0)
    })

    # Overall summary
    overall = {
        "mAP (macro mean AP)": map_macro,
        "AP (micro)": ap_micro,
        "AUROC (macro mean)": auroc_macro,
        "AUROC (micro)": auroc_micro,
        "Precision (micro)@{:.2f}".format(threshold): p_micro,
        "Recall (micro)@{:.2f}".format(threshold): r_micro,
        "F1 (micro)@{:.2f}".format(threshold): f1_micro,
        "Precision (macro)@{:.2f}".format(threshold): p_macro,
        "Recall (macro)@{:.2f}".format(threshold): r_macro,
        "F1 (macro)@{:.2f}".format(threshold): f1_macro,
        "Subset Accuracy": subset_acc,
        "Hamming Loss": hamming,
    }

    overall_df = pd.DataFrame(list(overall.items()), columns=["Metric", "Value"])
    return per_class_df, overall_df, y_pred

per_class_df, overall_df, y_pred = multilabel_metrics(y_true, y_scores, class_names, threshold=0.5)

print("\n=== Overall Metrics ===")
print(overall_df.to_string(index=False))

print("\n=== Per-class Metrics ===")
print(per_class_df.sort_values("AP", ascending=False).to_string(index=False))


Using logic model for evaluation...


Predicting: 100%|██████████| 364/364 [01:05<00:00,  5.55it/s]



=== Overall Metrics ===
                Metric    Value
   mAP (macro mean AP) 0.876869
            AP (micro) 0.924005
    AUROC (macro mean) 0.976367
         AUROC (micro) 0.982715
Precision (micro)@0.50 0.914179
   Recall (micro)@0.50 0.778051
       F1 (micro)@0.50 0.840640
Precision (macro)@0.50 0.908880
   Recall (macro)@0.50 0.671053
       F1 (macro)@0.50 0.754795
       Subset Accuracy 0.534604
          Hamming Loss 0.030427

=== Per-class Metrics ===
      Class       AP    AUROC  Precision@0.50  Recall@0.50  F1@0.50  Support
     animal 0.993604 0.994719        0.990977     0.944126 0.966985     2094
  aeroplane 0.986431 0.998658        0.996377     0.801749 0.888530      343
    vehicle 0.982152 0.987750        0.942974     0.936773 0.939863     1977
        cat 0.969320 0.991171        0.956088     0.885397 0.919386      541
      train 0.958829 0.995697        0.899628     0.892989 0.896296      271
     person 0.957263 0.967328        0.951559     0.816531 0.878889   

## Constraints at Evaluation 

In [ ]:
from ccn.constraints_layer import ConstraintsLayer
from ccn.constraints_group import ConstraintsGroup
from ccn.constraint import Constraint
from ccn.clauses_group import ClausesGroup

import torch

In [13]:
rules_path = "rules.txt"
group   = ConstraintsGroup(rules_path)
clauses = ClausesGroup.from_constraints_group(group)
layer   = ConstraintsLayer.from_clauses_group(clauses, num_classes=23, centrality="katz")

<generator object ConstraintsGroup.__init__.<locals>.<genexpr> at 0x000001ECBC79CF20>


In [ ]:
import os
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_fscore_support, f1_score, precision_score, recall_score,
    accuracy_score, hamming_loss
)

# ---- Inputs you already have ----
# class_names = [...]
# val_images, val_labels, val_labels_obs = np.load(...)
# transform, l_model defined elsewhere
print("Using logic model for evaluation...")
IMG_DIR = r"data/pascal/VOCdevkit/VOC2012/JPEGImages/"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
l_model = l_model.to(DEVICE).eval()

# ---------- 1) Get probabilities for ALL validation images ----------
def predict_all_probs(img_names, batch_size=16, threshold=None):
    """
    Returns:
        y_scores: (N, C) float array of probabilities
    """
    N = len(img_names)
    # Peek one image to get C
    first_img = Image.open(os.path.join(IMG_DIR, img_names[0])).convert("RGB")
    with torch.no_grad():
        C = l_model(transform(first_img).unsqueeze(0).to(DEVICE)).shape[-1]

    y_scores = np.zeros((N, C), dtype=np.float32)

    # Simple batching without DataLoader
    for start in tqdm(range(0, N, batch_size), desc="Predicting"):
        end = min(start + batch_size, N)
        batch_imgs = []
        for i in range(start, end):
            p = os.path.join(IMG_DIR, img_names[i])
            img = Image.open(p).convert("RGB")
            batch_imgs.append(transform(img))
        batch_tensor = torch.stack(batch_imgs, dim=0).to(DEVICE)

        with torch.no_grad():
            logits = l_model(batch_tensor)
            probs = torch.sigmoid(logits).cpu().numpy()
        y_scores[start:end] = probs

    return y_scores

# Ensure filenames are strings (sometimes np.load yields bytes)
val_files = [fn.decode() if isinstance(fn, bytes) else str(fn) for fn in val_images]
y_true = val_labels.astype(int)             # shape (N, C)
y_scores = predict_all_probs(val_files, batch_size=16)

# ---------- 2) Compute metrics ----------
def multilabel_metrics(y_true, y_scores, class_names, threshold=0.5):
    """
    Computes:
      - Per-class: AP, AUROC, Precision, Recall, F1 (at threshold)
      - Overall: mAP (macro), AP-micro, AUROC-macro, AUROC-micro,
                 Precision/Recall/F1 (micro & macro at threshold),
                 Subset Accuracy, Hamming Loss
    """
    N, C = y_true.shape
    assert C == len(class_names), "class_names length must match labels"

    # Per-class AP & AUROC (use probabilities)
    ap_per_class = []
    auroc_per_class = []
    for c in range(C):
        y_c = y_true[:, c]
        s_c = y_scores[:, c]
        # average_precision_score works even with class imbalance
        ap_c = average_precision_score(y_c, s_c) if (y_c.max() != y_c.min()) else np.nan
        # roc_auc_score requires both classes present
        try:
            auc_c = roc_auc_score(y_c, s_c)
        except ValueError:
            auc_c = np.nan
        ap_per_class.append(ap_c)
        auroc_per_class.append(auc_c)

    # Threshold to get binary predictions
    y_pred = (y_scores >= threshold).astype(int)

    # Per-class P/R/F1 at threshold
    prec_c, rec_c, f1_c, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    # Overall (micro/macro)
    map_macro = np.nanmean(ap_per_class)
    ap_micro  = average_precision_score(y_true, y_scores, average="micro")
    auroc_macro = np.nanmean(auroc_per_class)
    # For AUROC micro, flatten
    try:
        auroc_micro = roc_auc_score(y_true.ravel(), y_scores.ravel())
    except ValueError:
        auroc_micro = np.nan

    p_micro = precision_score(y_true, y_pred, average="micro", zero_division=0)
    r_micro = recall_score(y_true, y_pred, average="micro", zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)

    p_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    r_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

    subset_acc = accuracy_score(y_true, y_pred)  # exact match ratio
    hamming = hamming_loss(y_true, y_pred)

    # Build per-class table
    per_class_df = pd.DataFrame({
        "Class": class_names,
        "AP": ap_per_class,
        "AUROC": auroc_per_class,
        "Precision@{:.2f}".format(threshold): prec_c,
        "Recall@{:.2f}".format(threshold): rec_c,
        "F1@{:.2f}".format(threshold): f1_c,
        "Support": y_true.sum(axis=0)
    })

    # Overall summary
    overall = {
        "mAP (macro mean AP)": map_macro,
        "AP (micro)": ap_micro,
        "AUROC (macro mean)": auroc_macro,
        "AUROC (micro)": auroc_micro,
        "Precision (micro)@{:.2f}".format(threshold): p_micro,
        "Recall (micro)@{:.2f}".format(threshold): r_micro,
        "F1 (micro)@{:.2f}".format(threshold): f1_micro,
        "Precision (macro)@{:.2f}".format(threshold): p_macro,
        "Recall (macro)@{:.2f}".format(threshold): r_macro,
        "F1 (macro)@{:.2f}".format(threshold): f1_macro,
        "Subset Accuracy": subset_acc,
        "Hamming Loss": hamming,
    }

    overall_df = pd.DataFrame(list(overall.items()), columns=["Metric", "Value"])
    return per_class_df, overall_df, y_pred

per_class_df, overall_df, y_pred = multilabel_metrics(y_true, y_scores, class_names, threshold=0.5)

print("\n=== Overall Metrics ===")
print(overall_df.to_string(index=False))

print("\n=== Per-class Metrics ===")
print(per_class_df.sort_values("AP", ascending=False).to_string(index=False))


In [2]:
import numpy as np 

In [3]:
llll = np.load(r"data\pascal\formatted_train_images.npy")

In [4]:
llll.shape

(5717,)

In [7]:
ppp = np.load(r"data\pascal\formatted_val_labels.npy")

In [ ]:
ppp.shape

(5823, 23)

: 